# Code Complexity vs Agent Resolution Rate

**Question**: Does structural code complexity predict whether coding agents can fix bugs?

**Finding**: No. File-level complexity metrics do not predict agent success. Only patch-level metrics (what the agent must produce) correlate significantly. The further you zoom out from the actual change, the weaker the signal.

Metrics are analyzed at three granularity levels:
1. **Patch-level** — size and scatter of the gold patch (LOC, files, hunks)
2. **Function-level** — complexity of the specific function(s) being modified
3. **File-level** — whole-file structural complexity (cyclomatic, Halstead, MI, etc.)

In [1]:
import json
import math
import sys
import warnings
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from unidiff import PatchSet

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=stats.ConstantInputWarning)

from utils import DATASET_PATH, count_files_in_patch, count_loc_in_patch, load_aggregate_results  # noqa: E402

from bcbench.dataset import load_dataset_entries  # noqa: E402
from bcbench.results.metrics import pass_hat_k  # noqa: E402

METRICS_PATH = DATASET_PATH.parent / "bcbench_metrics.jsonl"
FN_METRICS_PATH = DATASET_PATH.parent / "bcbench_fn_metrics.json"
ANALYSIS_SETUP = "copilot-opus-4-6"

In [2]:
# --- Resolution rates ---
runs_df = load_aggregate_results(category="bug-fix")[ANALYSIS_SETUP]
resolution_df = runs_df.groupby("instance_id")["resolved"].agg(mean_rate="mean", n_runs="count", n_resolved="sum").reset_index()
resolution_df["pass_hat_5"] = resolution_df.apply(lambda r: pass_hat_k(int(r["n_runs"]), int(r["n_resolved"]), 5), axis=1)

# --- Features ---
dataset_entries = load_dataset_entries(DATASET_PATH)
entries_raw = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines()]
entries_by_id = {e["instance_id"]: e for e in entries_raw}

patch_rows = []
for e_raw, e in zip(entries_raw, dataset_entries, strict=False):
    ps = PatchSet(e_raw["patch"])
    patch_rows.append(
        {
            "instance_id": e.instance_id,
            "patch_loc": count_loc_in_patch(e.patch),
            "patch_files": count_files_in_patch(e.patch),
            "total_hunks": sum(len(list(pf)) for pf in ps),
        }
    )
patch_df = pd.DataFrame(patch_rows)

file_rows = []
for line in METRICS_PATH.read_text(encoding="utf-8").splitlines():
    entry = json.loads(line)
    all_m = [f["metrics"] for f in entry["files"] if f["metrics"]]
    if not all_m:
        file_rows.append({"instance_id": entry["instance_id"]})
        continue
    total_sloc = sum(m["loc"]["sloc"] for m in all_m) or 1
    patch_lines = sum(pf.added + pf.removed for pf in PatchSet(entries_by_id[entry["instance_id"]]["patch"]))
    file_rows.append(
        {
            "instance_id": entry["instance_id"],
            "file_cyclomatic": sum(m["cyclomatic"]["sum"] for m in all_m),
            "file_cognitive": sum(m["cognitive"]["sum"] for m in all_m),
            "file_sloc": sum(m["loc"]["sloc"] for m in all_m),
            "file_halstead_volume": sum(m["halstead"]["volume"] for m in all_m),
            "file_halstead_bugs": sum(m["halstead"]["bugs"] for m in all_m),
            "file_mi_vs": sum(m["mi"]["mi_visual_studio"] * m["loc"]["sloc"] for m in all_m) / total_sloc,
            "file_num_functions": sum(m["nom"]["functions"] for m in all_m),
            "patch_pct_of_file": patch_lines / total_sloc * 100,
        }
    )
file_df = pd.DataFrame(file_rows).fillna(0)
fn_df = pd.DataFrame(json.loads(FN_METRICS_PATH.read_text(encoding="utf-8")))

df = resolution_df.merge(patch_df, on="instance_id").merge(file_df, on="instance_id").merge(fn_df, on="instance_id", how="left")
print(f"{ANALYSIS_SETUP}: {len(df)} instances, {resolution_df['n_runs'].iloc[0]} runs each, {df['fn_cyclomatic_sum'].notna().sum()} with function-level metrics")

copilot-opus-4-6: 101 instances, 5 runs each, 81 with function-level metrics


## 1. Granularity Mismatch

In most tasks, the gold patch modifies less than 1% of the file. File-level complexity is noise.

In [3]:
pct = df["patch_pct_of_file"]
print(f"Patch as % of file SLOC:  median={pct.median():.1f}%  mean={pct.mean():.1f}%")
print(f"  <1%: {(pct < 1).sum()}/{len(pct)} ({(pct < 1).mean():.0%})   <5%: {(pct < 5).sum()}/{len(pct)} ({(pct < 5).mean():.0%})   <10%: {(pct < 10).sum()}/{len(pct)} ({(pct < 10).mean():.0%})")

Patch as % of file SLOC:  median=0.8%  mean=2.6%
  <1%: 57/101 (56%)   <5%: 88/101 (87%)   <10%: 94/101 (93%)


## 2. Correlation: Mean Resolution Rate (Spearman)

Each instance has a resolution rate from 5 runs (0/5, 1/5, ..., 5/5). We compute Spearman rank correlation between each metric and this rate across all instances. Spearman is non-parametric and handles the discretized outcome well.

In [4]:
TIERS = [
    (
        "Patch-level",
        [
            ("total_hunks", "Number of Hunks"),
            ("patch_loc", "Lines Changed (LOC)"),
            ("patch_files", "Files Changed"),
            ("patch_pct_of_file", "Patch % of File"),
        ],
    ),
    (
        "Function-level",
        [
            ("fn_cyclomatic_sum", "Cyclomatic (patched fn)"),
            ("fn_cognitive_sum", "Cognitive (patched fn)"),
            ("fn_sloc_sum", "SLOC (patched fn)"),
            ("fn_halstead_volume_sum", "Halstead Volume (patched fn)"),
            ("fn_halstead_effort_sum", "Halstead Effort (patched fn)"),
        ],
    ),
    (
        "File-level",
        [
            ("file_cyclomatic", "Cyclomatic (whole file)"),
            ("file_cognitive", "Cognitive (whole file)"),
            ("file_sloc", "SLOC (whole file)"),
            ("file_halstead_volume", "Halstead Volume (whole file)"),
            ("file_halstead_bugs", "Halstead Est. Bugs (whole file)"),
            ("file_mi_vs", "Maintainability Index (whole file)"),
            ("file_num_functions", "Num Functions (whole file)"),
        ],
    ),
]

corr_rows = []
for tier_name, metrics in TIERS:
    for col, label in metrics:
        valid = df[[col, "mean_rate"]].dropna()
        if len(valid) < 10:
            continue
        rho, p = stats.spearmanr(valid[col], valid["mean_rate"])
        if math.isnan(rho):
            continue
        corr_rows.append({"tier": tier_name, "column": col, "metric": label, "spearman_rho": round(rho, 3), "p_value": round(p, 4), "significant": p < 0.05, "n": len(valid)})

corr_df = pd.DataFrame(corr_rows)

for tier in ["Patch-level", "Function-level", "File-level"]:
    subset = corr_df[corr_df["tier"] == tier].sort_values("p_value")
    print(f"\n{tier.upper()}")
    for _, r in subset.iterrows():
        sig = "**" if r["p_value"] < 0.01 else "* " if r["significant"] else "  "
        print(f"  {sig} {r['metric']:40s} \u03c1={r['spearman_rho']:+.3f}  p={r['p_value']:.4f}  (n={r['n']})")


PATCH-LEVEL
  ** Number of Hunks                          ρ=-0.332  p=0.0007  (n=101)
  ** Lines Changed (LOC)                      ρ=-0.323  p=0.0010  (n=101)
  ** Files Changed                            ρ=-0.266  p=0.0071  (n=101)
  *  Patch % of File                          ρ=-0.208  p=0.0371  (n=101)

FUNCTION-LEVEL
     Cyclomatic (patched fn)                  ρ=-0.151  p=0.1786  (n=81)
     Cognitive (patched fn)                   ρ=-0.148  p=0.1878  (n=81)
     SLOC (patched fn)                        ρ=-0.119  p=0.2909  (n=81)
     Halstead Volume (patched fn)             ρ=-0.113  p=0.3156  (n=81)
     Halstead Effort (patched fn)             ρ=-0.065  p=0.5638  (n=81)

FILE-LEVEL
     Maintainability Index (whole file)       ρ=-0.161  p=0.1085  (n=101)
     Cognitive (whole file)                   ρ=-0.082  p=0.4122  (n=101)
     Halstead Est. Bugs (whole file)          ρ=-0.078  p=0.4364  (n=101)
     Halstead Volume (whole file)             ρ=-0.068  p=0.5020  (n=101)
  

In [5]:
tier_colors = {"Patch-level": "#2ecc71", "Function-level": "#f39c12", "File-level": "#e74c3c"}
plot_df = corr_df.sort_values("spearman_rho")

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=[r["metric"] for _, r in plot_df.iterrows()],
        x=plot_df["spearman_rho"],
        orientation="h",
        marker_color=[tier_colors[t] for t in plot_df["tier"]],
        text=[f"\u03c1={r['spearman_rho']:.3f}{'*' if r['significant'] else ''}" for _, r in plot_df.iterrows()],
        textposition="outside",
        showlegend=False,
    )
)
for tier, color in tier_colors.items():
    fig.add_trace(go.Bar(y=[None], x=[None], orientation="h", marker_color=color, name=tier, showlegend=True))

fig.update_layout(
    title=f"Spearman \u03c1 with Mean Resolution Rate ({ANALYSIS_SETUP})",
    xaxis_title="Spearman \u03c1",
    height=550,
    width=900,
    margin={"l": 300},
    xaxis={"range": [-0.45, 0.15], "zeroline": True, "zerolinecolor": "black", "zerolinewidth": 1},
    legend={"x": 0.01, "y": 0.01},
)
fig.show()

## 3. Robustness Check: pass\u0302\u2075 (Mann-Whitney U)

pass\u0302\u2075 = C(c, 5) / C(n, 5) is binary when n=k=5: 1 if all 5 runs succeed, 0 otherwise. This asks a stricter question: *is the task reliably solvable?* For a binary outcome, we use the Mann-Whitney U test, comparing metric distributions between the always-solved and not-always-solved groups.

In [6]:
all_metrics = [(t, c, label) for t, ms in TIERS for c, label in ms]

mw_rows = []
for tier, col, label in all_metrics:
    valid = df[[col, "pass_hat_5"]].dropna()
    group_fail = valid[valid["pass_hat_5"] == 0][col]
    group_pass = valid[valid["pass_hat_5"] == 1][col]
    if len(group_fail) < 5 or len(group_pass) < 5:
        continue
    u_stat, p_val = stats.mannwhitneyu(group_fail, group_pass, alternative="two-sided")
    r_rb = 1 - (2 * u_stat) / (len(group_fail) * len(group_pass))
    mw_rows.append(
        {"tier": tier, "metric": label, "median_fail": group_fail.median(), "median_pass": group_pass.median(), "p_value": round(p_val, 4), "r_rb": round(r_rb, 3), "significant": p_val < 0.05}
    )

mw_df = pd.DataFrame(mw_rows)
n_fail = int((df["pass_hat_5"] == 0).sum())
n_pass = int((df["pass_hat_5"] == 1).sum())
print(f"pass\u0302\u2075 groups: always-solved={n_pass}, not-always={n_fail}\n")

for tier in ["Patch-level", "Function-level", "File-level"]:
    subset = mw_df[mw_df["tier"] == tier].sort_values("p_value")
    print(f"{tier.upper()}")
    for _, r in subset.iterrows():
        sig = "* " if r["significant"] else "  "
        print(f"  {sig}{r['metric']:40s} p={r['p_value']:.4f}  r={r['r_rb']:+.3f}  median: {r['median_fail']:.0f} vs {r['median_pass']:.0f}")
    print()

pasŝ⁵ groups: always-solved=51, not-always=50

PATCH-LEVEL
  * Number of Hunks                          p=0.0082  r=-0.291  median: 2 vs 1
  * Lines Changed (LOC)                      p=0.0088  r=-0.302  median: 16 vs 5
  * Files Changed                            p=0.0158  r=-0.190  median: 1 vs 1
    Patch % of File                          p=0.1528  r=-0.165  median: 1 vs 1

FUNCTION-LEVEL
    Cognitive (patched fn)                   p=0.1341  r=-0.194  median: 9 vs 4
    Cyclomatic (patched fn)                  p=0.1379  r=-0.192  median: 7 vs 5
    Halstead Volume (patched fn)             p=0.1468  r=-0.188  median: 657 vs 399
    SLOC (patched fn)                        p=0.1793  r=-0.174  median: 33 vs 20
    Halstead Effort (patched fn)             p=0.3470  r=-0.122  median: 5597 vs 3022

FILE-LEVEL
  * Maintainability Index (whole file)       p=0.0495  r=-0.193  median: 0 vs 0
    Cognitive (whole file)                   p=0.1711  r=-0.158  median: 142 vs 97
    Halstead Est

## 4. Summary

In [7]:
spearman_sig = corr_df.groupby("tier")["significant"].agg(["sum", "count"])
mw_sig = mw_df.groupby("tier")["significant"].agg(["sum", "count"])

print("Significant metrics by tier:")
print(f"{'Tier':<20s} {'Spearman (mean rate)':<25s} {'Mann-Whitney (pass^5)'}")
print("-" * 65)
for tier in ["Patch-level", "Function-level", "File-level"]:
    s = spearman_sig.loc[tier]
    m = mw_sig.loc[tier]
    print(f"{tier:<20s} {int(s['sum'])}/{int(s['count']):<24} {int(m['sum'])}/{int(m['count'])}")

print(f"\nIn {(df['patch_pct_of_file'] < 1).mean():.0%} of tasks, the patch touches <1% of the file.")
print("File-level complexity is noise. Agent difficulty is dominated by the")
print("search problem (finding where to fix), not structural complexity.")
print("\nNumber of hunks is the best predictor \u2014 it captures how many")
print("distinct locations the agent must correctly identify and modify.")

Significant metrics by tier:
Tier                 Spearman (mean rate)      Mann-Whitney (pass^5)
-----------------------------------------------------------------
Patch-level          4/4                        3/4
Function-level       0/5                        0/5
File-level           0/7                        1/7

In 56% of tasks, the patch touches <1% of the file.
File-level complexity is noise. Agent difficulty is dominated by the
search problem (finding where to fix), not structural complexity.

Number of hunks is the best predictor — it captures how many
distinct locations the agent must correctly identify and modify.
